In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import BooleanType
import pyspark.sql.functions as f

spark = SparkSession.builder.appName("NYC Taxi Data").getOrCreate()

taxi_df = spark.read.table("nyc_bronze.bronze.yello_taxi_raw")

df = taxi_df.toDF(*[c.lower() for c in taxi_df.columns]).withColumnsRenamed({"vendorid":"vendor_id", "ratecodeid":"ratecode_id", "pulocationid":"pulocation_id","dolocationid":"dolocation_id"}).filter(f.col('total_amount') >= 0)

transformed_df = df.withColumn('is_incomplete_record', f.when((f.col('payment_type') == 0), True).otherwise(False))
transformed_df = transformed_df.filter((f.col("tpep_pickup_datetime") >= "2026-01-01") & (f.col("tpep_pickup_datetime") < "2026-06-01"))
transformed_df.write.mode("overwrite").saveAsTable("nyc_silver.yellow_taxi_silver.yellow_taxi_transformed")